# DASCH Coadd Prototype

This notebook reproduces the main steps of the DASCH median coadd pipeline
for a 5 deg² tile centered on M31. It is intended for local experimentation
and demonstration. Run inside the Docker container or a local environment
with the packages from `docker/requirements.txt` installed.

Pipeline overview:
1. Load tile config (`config/m31.json`)
2. Build output WCS grid
3. Load / simulate input plate FITS files
4. Compute per-plate weight maps
5. Reproject plates onto the common grid
6. Median coadd
7. Source detection with SEP
8. Write outputs and QA plots

In [ ]:
import sys, os
# Make sure the repo root is on the path when running in the container
repo_root = '/work'
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u

## 1. Load config

In [ ]:
config_path = Path('/work/config/m31.json')
with open(config_path) as f:
    cfg = json.load(f)
print(json.dumps(cfg, indent=2))

## 2. Build output WCS

In [ ]:
import math

ra0  = cfg['center_ra_deg']
dec0 = cfg['center_dec_deg']
scale_deg = cfg['pixel_scale_arcsec'] / 3600.0
side_deg  = math.sqrt(cfg['area_sqdeg'])
n_pix     = int(side_deg / scale_deg)
if n_pix % 2:
    n_pix += 1

wcs_out = WCS(naxis=2)
wcs_out.wcs.crpix = [n_pix/2+0.5, n_pix/2+0.5]
wcs_out.wcs.cdelt = [-scale_deg,  scale_deg]
wcs_out.wcs.crval = [ra0, dec0]
wcs_out.wcs.ctype = ['RA---TAN', 'DEC--TAN']

print(f'Output grid: {n_pix} x {n_pix} pixels  ({cfg["pixel_scale_arcsec"]} arcsec/pix)')
print(wcs_out)

## 3. Generate synthetic test plates

In RUN_MODE=test the pipeline generates synthetic plates. Here we do the same manually so you can inspect individual plates.

In [ ]:
from pipeline.run_coadd import _make_synthetic_plate

plates = []
for i in range(6):
    data, wcs, mask = _make_synthetic_plate(cfg, plate_id=i, seed=42+i)
    plates.append((f'SYN{i:04d}', data, wcs, mask))
    print(f'Plate SYN{i:04d}: shape={data.shape}, mask_frac={mask.mean():.3f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, (pid, data, wcs, mask) in zip(axes.flat, plates):
    vmin, vmax = np.nanpercentile(data, [1, 99])
    ax.imshow(data, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
    ax.set_title(pid)
    ax.axis('off')
fig.suptitle('Synthetic input plates')
plt.tight_layout()
plt.show()

## 4. Compute weight maps

In [ ]:
from pipeline.run_coadd import compute_weight

weights = []
for pid, data, wcs, mask in plates:
    w = compute_weight(data, mask)
    weights.append(w)
    print(f'{pid}: weight range [{w.min():.2e}, {w.max():.2e}]')

## 5. Reproject onto common grid

In [ ]:
from pipeline.run_coadd import reproject_plate

tmp_dir = Path('/work/tmp')
repr_stack, mask_stack, weight_stack, plate_ids = [], [], [], []

for (pid, data, wcs, mask), w in zip(plates, weights):
    r_data, r_mask, r_weight = reproject_plate(
        data, wcs, mask, w, wcs_out, (n_pix, n_pix), pid, tmp_dir
    )
    repr_stack.append(r_data)
    mask_stack.append(r_mask)
    weight_stack.append(r_weight)
    plate_ids.append(pid)
    print(f'{pid}: finite_frac={np.isfinite(r_data).mean():.3f}')

## 6. Median coadd

In [ ]:
from pipeline.run_coadd import median_coadd

coadd, weight_map, mask_map, contributors = median_coadd(repr_stack, mask_stack, weight_stack)

print(f'Coadd shape   : {coadd.shape}')
print(f'Finite pixels : {np.isfinite(coadd).sum()} / {coadd.size}')
print(f'Max contributors per pixel: {contributors.max()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

vmin, vmax = np.nanpercentile(coadd[np.isfinite(coadd)], [1, 99])
axes[0].imshow(coadd, origin='lower', cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title('Median coadd')

axes[1].imshow(contributors, origin='lower', cmap='viridis')
axes[1].set_title('Contributors map')

axes[2].imshow(mask_map, origin='lower', cmap='Reds', vmin=0, vmax=1)
axes[2].set_title('Mask map')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Source detection

In [ ]:
from pipeline.run_coadd import detect_sources

catalog = detect_sources(coadd, wcs_out)
print(f'Detected {len(catalog)} sources')
if len(catalog) > 0:
    print(catalog[:5])

## 8. Write outputs

In [ ]:
from pipeline.run_coadd import write_outputs, write_qa_plots

out_dir = Path('/work/notebook_output')
write_outputs(coadd, weight_map, mask_map, contributors, catalog, wcs_out, cfg, plate_ids, out_dir)
write_qa_plots(coadd, contributors, plate_ids, out_dir)
print('Output files:')
for f in sorted(out_dir.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size} bytes)')